# الحرارة اليومية وزيارات عيادات المواقع

معهد هَجير لبحوث الصحة المهنية (افتراضي). الحرارة: سلسلة عامة حقيقية من NASA POWER. الزيارات: بيانات اصطناعية من صنع المقرر.

الدفتر المرجعي: ينفذ خطة التحليل المعلنة في `analysis_plan.txt` كما هي، ويعمل من تشغيل نظيف: أعد تشغيل النواة وشغّل كل الخلايا.

In [1]:
import hashlib
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd

# the two data files sit beside the notebook (in a hosted notebook service: upload them first)
DATA = next((p for p in (Path("."), Path("files/analysis"), Path("../files/analysis"))
             if (p / "power_daily_2024_raw.csv").exists()), None)
assert DATA is not None, "put power_daily_2024_raw.csv and clinic_visits_synthetic.csv beside the notebook"
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.5.5 | pandas 3.1.1


In [2]:
# the public NASA POWER file, exactly as downloaded: a header block, then the table
lines = (DATA / "power_daily_2024_raw.csv").read_text(encoding="utf-8").splitlines()
start = next(i for i, line in enumerate(lines) if line.startswith("-END HEADER-")) + 1
weather = pd.read_csv(io.StringIO("\n".join(lines[start:])))
weather["date"] = pd.to_datetime(weather["YEAR"].astype(str) + weather["DOY"].astype(str).str.zfill(3), format="%Y%j")
assert (weather[["T2M_MAX", "T2MWET"]] > -900).all().all(), "missing values (-999) in the weather file"
print(len(weather), "days |", weather["date"].min().date(), "to", weather["date"].max().date())

153 days | 2024-05-01 to 2024-09-30


In [3]:
# synthetic counts written for this course: workers on site and heat-related clinic visits per day
clinic = pd.read_csv(DATA / "clinic_visits_synthetic.csv", parse_dates=["date"])
df = weather.merge(clinic, on="date", how="inner", validate="one_to_one")
print(len(df), "days after the merge |", int(df["heat_visits"].sum()), "visits")

153 days after the merge | 351 visits


## الخطة

العتبة والاستبعاد والبذرة من الخطة المعلنة، في خلية واحدة ظاهرة.

In [4]:
# the pre-stated plan, items 3, 5 and 6: nothing below is chosen after seeing the visits
THRESHOLD = 45.0      # hot day: T2M_MAX of the same day at or above this (plan, item 3)
MIN_WORKERS = 200     # days with fewer workers on site are excluded (plan, item 5)
N_BOOT = 2000         # bootstrap resamples of days (plan, item 6)
SEED = 208            # fixed seed (plan, item 6)

In [5]:
before = len(df)
df = df[df["workers_on_site"] >= MIN_WORKERS].reset_index(drop=True)
print("excluded", before - len(df), "days with fewer than", MIN_WORKERS, "workers |", len(df), "days analysed")

excluded 22 days with fewer than 200 workers | 131 days analysed


In [6]:
df["hot"] = df["T2M_MAX"] >= THRESHOLD
print(int(df["hot"].sum()), "hot days |", int((~df["hot"]).sum()), "other days")

31 hot days | 100 other days


In [7]:
def rate_ratio(d):
    """Visits per 1,000 worker-days on hot days and on other days, and their ratio."""
    hot, other = d[d["hot"]], d[~d["hot"]]
    r_hot = 1000 * hot["heat_visits"].sum() / hot["workers_on_site"].sum()
    r_other = 1000 * other["heat_visits"].sum() / other["workers_on_site"].sum()
    return r_hot, r_other, r_hot / r_other

r_hot, r_other, rr = rate_ratio(df)
print(f"hot days {r_hot:.2f} per 1,000 worker-days | other days {r_other:.2f} | rate ratio {rr:.2f}")

hot days 2.66 per 1,000 worker-days | other days 2.04 | rate ratio 1.31


In [8]:
rng = np.random.default_rng(SEED)
n = len(df)
boot = []
for _ in range(N_BOOT):
    boot.append(rate_ratio(df.iloc[rng.integers(0, n, n)])[2])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"rate ratio {rr:.2f} (95% CI {lo:.2f} to {hi:.2f}), {N_BOOT} resamples, seed {SEED}")

rate ratio 1.31 (95% CI 0.95 to 1.71), 2000 resamples, seed 208


In [9]:
results = {"days": int(n), "hot_days": int(df["hot"].sum()), "rate_hot": round(float(r_hot), 2),
           "rate_other": round(float(r_other), 2), "rate_ratio": round(float(rr), 2),
           "ci_low": round(float(lo), 2), "ci_high": round(float(hi), 2)}
fingerprint = hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:12]
print(json.dumps(results))
print("fingerprint", fingerprint)

{"days": 131, "hot_days": 31, "rate_hot": 2.66, "rate_other": 2.04, "rate_ratio": 1.31, "ci_low": 0.95, "ci_high": 1.71}
fingerprint 8e921892452f


## التحليل الثانوي المعلن

يُعرض بجوار النتيجة الأساسية، لا بدلًا منها.

In [10]:
# secondary analysis declared in the plan, item 7: the temperature of the day before, same threshold
lag = df.copy()
lag["hot"] = lag["T2M_MAX"].shift(1) >= THRESHOLD
lag = lag.iloc[1:]
s_hot, s_other, s_rr = rate_ratio(lag)
print(f"secondary (previous day): rate ratio {s_rr:.2f}, reported as secondary")

secondary (previous day): rate ratio 1.40, reported as secondary


## جملة النتيجة

في 131 يومًا، كان معدل زيارات عيادات المواقع بسبب أمراض الحرارة 2.66 لكل ألف يوم عمل في الأيام الحارة (31 يومًا) و2.04 في غيرها. نسبة المعدل 1.31 (فترة الثقة 95%: من 0.95 إلى 1.71). الزيارات بيانات اصطناعية، والنتيجة ارتباط لا سببية.